# 02 · Limpieza de datos

Preparo los datos para el modelo de riesgo de lesión. Parto de los CSV originales de `data/raw/`, aplico las decisiones tomadas en la exploración (`01-exploracion-datasetNBA.ipynb`) y guardo una tabla limpia en `data/nba_load.parquet`, que es la que carga `project-model.ipynb`.

Cada paso queda registrado con el número de filas que afecta.

In [3]:
#importación de herramientas y librerías necesarias para el análisis de datos
from pathlib import Path
import numpy as np #librería de cálculo númerico sobre la que funciona pandas
import pandas as pd

RAW = Path("..")/"data"/"raw"
OUT = Path("..")/"data"/"nba_load.parquet" #es donde se guardará el fichero limpio en project/data 
pd.set_option("display.max_columns", None)

In [4]:
#Cargar el fichero de datos en bruto
player_stats = pd.read_csv(RAW/"playerStatistics.csv", low_memory=False) #low_memory=False para evitar el warning de memoria
players = pd.read_csv(RAW/"players.csv", low_memory=False)

#crear lista vacía y en cada paso se va a apuntar una línea con tres datos:
#- que se hizo, - cuantas filas afectó, - por qué se hizo. 
informe =[] #Esta es la lista vacía

print("Filas iniciales de player_stats:", len(player_stats))

Filas iniciales de player_stats: 1669922


In [6]:
#Empezamos a limpiar el fechero y a filtrar los datos que me interesan

#PRIMERO 
#Fechas: solo quiero desde la temporada 2015-16 
player_stats["gameDate"] = pd.to_datetime(player_stats["gameDate"], errors="coerce") #convierto la columna de fechas a tipo fecha, si no puede convertirlo lo pone como NaT
n = len(player_stats) #guardo el número de filas inicial
player_stats = player_stats[player_stats["gameDate"] >= "2015-10-01"] #filtro las filas que cumplen la condición

#Añado a la lista de informe lo que se ha hecho:
informe.append(["Filtrado de fechas: partidos anteriores al 01/10/2015", n-len(player_stats), "Se han eliminado los partidos anteriores a la temporada 2015-16 (fuera de mi ventana de trabajo)"])
#-----------------------------------------------------------

#SEGUNDO
#Tipo de partido: solo quiero partidos de temporada regular y  playoffs. 
#Voy a unificar los nombres de los tipos de partido y quedarme con los que me interesen. 
mapa_tipo_partido = {
    "Regular Season": "Regular Season", "Playoffs": "Playoffs", "Play-in Tournament": "Play-in",
    "NBA Emirates Cup": "NBA Cup", "Emirates NBA Cup": "NBA Cup", "NBA Cup": "NBA Cup",
    "in-season-knockout": "NBA Cup", "Preseason": "Preseason", "Pre Season": "Preseason",
    "All-Star Game": "All-Star",
}
player_stats["gameType"] = player_stats["gameType"].map(mapa_tipo_partido)
n = len(player_stats)
player_stats = player_stats[player_stats["gameType"].isin(["Regular Season", "Playoffs", "Play-in", "NBA Cup"])]

#Añado a la lista de informe lo que se ha hecho:
informe.append(("No he inlcuido Pretemporada, All-Star y otros", n - len(player_stats), "No generan carga competitiva real"))
#-----------------------------------------------------------

#TERCERO
#Jugadores sin indetificar
n=len(player_stats)
player_stats = player_stats.dropna(subset=["personId"]) #elimino las filas que no tienen personId

#Añado a la lista de informe lo que se ha hecho:
informe.append(("Eliminación de jugadores sin personId", n - len(player_stats), "No se puede identificar al jugador, no se puede analizar"))

player_stats["personId"] = player_stats["personId"].astype(int) #convierto a int la columna personId para poder hacer el merge con players

#-----------------------------------------------------------

print("Filas finales de player_stats:", len(player_stats))
player_stats["gameType"].value_counts() #veo cuántas filas hay de cada tipo de partido

Filas finales de player_stats: 366218


gameType
Regular Season    338439
Playoffs           24959
NBA Cup             1822
Play-in              998
Name: count, dtype: int64